# Persona Simulation Exploration

## Exploratory Scenario Testing

This notebook separates the persona and LLM workflow from the main insight notebook so the public project stays focused. The purpose of this notebook is to explore how a product team might pressure-test interaction scenarios using synthetic inputs and structured prompting.

This workflow is framed as:

- exploratory scenario testing
- synthetic workflow simulation
- not a substitute for user research


## Setup

This notebook assumes `google-genai` is installed in the environment before running model calls. The helper module keeps the notebook readable by moving reusable functions out of the notebook body.


In [ ]:
import os
from pathlib import Path

import pandas as pd
from IPython.display import display

from habitbandz_portfolio_utils import (
    add_simulation_metrics,
    create_genai_client,
    flatten_simulation_results,
    get_user_examples,
    prepare_feedback_dataframe,
    run_simulation,
    summarize_by_scenario,
)

pd.set_option("display.max_colwidth", 120)
DATA_PATH = Path("data/habitbandz_synthetic_feedback.csv")


## Synthetic Data and Grounding Examples

The prompts below use synthetic participant language as grounding examples. This keeps the public workflow aligned with the original structure while avoiding private research data.


In [ ]:
df = prepare_feedback_dataframe(pd.read_csv(DATA_PATH))
user_examples = get_user_examples(df, limit=6)

display(df[["habit_desc", "triggers", "app_improve"]].head())
print("Example grounded language for prompts:")
for example in user_examples[:3]:
    print(f"- {example}")


## Simulation Assumptions

The point of this notebook is to explore scenario-level friction, not to infer real-world effectiveness. The product brief, personas, and scenarios are intentionally simplified so they are interpretable and easy to adapt.


In [ ]:
product_brief = """
Habit Bandz is a behavior-change product combining a wearable/button device, app, and dashboard.

Core workflow:
1. Onboarding
2. App setup
3. In-the-moment urge interruption
4. Self-monitoring
5. Reviewing patterns over time

Likely value:
- increases awareness of automatic habits
- provides tactile interruption during urge moments
- supports discreet use
- enables later reflection and interpretation

Known friction areas:
- logging must be fast enough during urge moments
- the wearable must be easy to find and activate without much visual attention
- dashboards need to feel interpretable and useful

Important:
This simulation is exploratory and should not be treated as user validation.
""".strip()

personas = {
    "Insight-Oriented": {
        "description": "More self-aware and motivated by tracking, reflection, and visible progress.",
        "needs": ["clear progress visibility", "timestamps and trends", "evidence that the product is working"],
        "pain_points": ["unclear dashboards", "weak interpretation support", "shallow data views"],
    },
    "Reactive": {
        "description": "Impulsive and stress-triggered, with limited bandwidth once the urge is active.",
        "needs": ["instant access", "minimal motor burden", "reliable tactile feedback"],
        "pain_points": ["too many steps", "slow interaction", "hesitation during escalation"],
    },
    "Overloaded": {
        "description": "Low bandwidth under stress or fatigue and easily overwhelmed by unclear flows.",
        "needs": ["simplicity", "clear next step", "low-friction interaction"],
        "pain_points": ["confusing navigation", "too much text", "setup overhead"],
    },
}

interaction_specs = {
    "wearable_button": {
        "device_location": "wrist-worn top surface",
        "button_location": "center of device face",
        "button_shape": "circular",
        "button_size": "thumb-sized",
        "button_height": "slightly raised",
        "press_confirmation": "haptic vibration",
        "requires_visual_attention": False,
        "usable_one_handed": True,
    },
    "app_logging": {
        "requires_phone": True,
        "estimated_steps": 3,
        "estimated_time_seconds": 8,
        "requires_visual_attention": True,
    },
    "monitoring_dashboard": {
        "shows_mood": True,
        "shows_urge_intensity": True,
        "shows_time_of_urge": True,
        "shows_multi_day_trends": "limited",
        "interpretation_support": "moderate",
    },
}

scenarios = [
    {
        "scenario_id": "bathroom_mirror_trigger",
        "context": {
            "location": "bathroom mirror",
            "attention_state": "visually locked on perceived imperfections",
            "stress_level": "moderate",
            "fatigue_level": "low",
            "phone_state": "not currently in hand",
            "habit_awareness": "partial",
        },
        "task": "The user begins scanning or picking and tries to interrupt the behavior using Habit Bandz.",
    },
    {
        "scenario_id": "late_night_scroll",
        "context": {
            "location": "in bed at night",
            "attention_state": "low and distracted by phone scrolling",
            "stress_level": "low to moderate",
            "fatigue_level": "high",
            "phone_state": "already in dominant hand",
            "habit_awareness": "low",
        },
        "task": "The user starts the habit absentmindedly while scrolling and must notice and respond quickly.",
    },
    {
        "scenario_id": "meeting_or_waiting_room",
        "context": {
            "location": "quiet social setting",
            "attention_state": "socially aware and self-conscious",
            "stress_level": "moderate",
            "fatigue_level": "low",
            "phone_state": "not appropriate to check openly",
            "habit_awareness": "high",
        },
        "task": "The user wants to interrupt the habit discreetly in public without drawing attention.",
    },
    {
        "scenario_id": "review_progress_after_several_days",
        "context": {
            "location": "home",
            "attention_state": "reflective",
            "stress_level": "low",
            "fatigue_level": "low",
            "phone_state": "available",
            "habit_awareness": "high",
        },
        "task": "The user reviews several days of data and tries to understand whether the product is helping.",
    },
]


## Model Configuration

Replace the placeholder below or set `GEMINI_API_KEY` in the environment before running. The notebook intentionally avoids storing a real key.


In [ ]:
API_KEY = os.getenv("GEMINI_API_KEY", "[INSERT KEY HERE]")
MODEL_NAME = "gemini-2.5-flash"

if API_KEY == "[INSERT KEY HERE]":
    client = None
    print("No API key found. Add GEMINI_API_KEY to the environment or replace the placeholder before running model calls.")
else:
    client = create_genai_client(API_KEY)
    print(f"Client ready for model: {MODEL_NAME}")


## Run Exploratory Scenario Testing

The code below loops over personas and scenarios, calls the helper-based simulation runner, then adds a few derived metrics to make the outputs easier to scan.


In [ ]:
results = []

if client is None:
    print("Skipping simulation because no API key was provided.")
else:
    for persona_name, persona in personas.items():
        for scenario in scenarios:
            results.append(
                run_simulation(
                    client=client,
                    persona_name=persona_name,
                    persona=persona,
                    scenario=scenario,
                    interaction_specs=interaction_specs,
                    examples=user_examples,
                    product_brief=product_brief,
                    model_name=MODEL_NAME,
                )
            )

results_df = add_simulation_metrics(flatten_simulation_results(results))
display(results_df.head())


## Scenario-Level Summary

This summary rolls the exploratory outputs up by scenario so it is easier to compare likely friction across contexts.


In [ ]:
scenario_summary = summarize_by_scenario(results_df)
display(scenario_summary)


## Interpretation Notes

- These outputs are exploratory and scenario-based.
- They are useful for narrowing questions, identifying friction, and deciding what to validate next.
- They should not be treated as evidence of real user behavior.
- The notebook is strongest when used to support product reasoning, not replace research.
